# NB45 — Android T21 QA
Tower 21: Tower of Android — Capacitor project structure, Gradle config, Android manifest, icons, splash screens, WebView setup, PWA manifest, responsive design, and Play Store readiness.

**Status:** SETUP_ONLY — implement after MVP desktop launch

**Scope:** File-based probes only. No servers, no HTTP requests, no module imports.

In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime
NORTHSTAR = "android-t21-qa"
NOTEBOOK_PATH = "notebooks/qa/45-android-t21-qa.ipynb"
TOWER = 21
TOWER_NAME = "Tower of Android"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
ANDROID = BROWSER_ROOT / "android"
ANDROID_APP = ANDROID / "app"
ANDROID_SRC = ANDROID_APP / "src" / "main"
ANDROID_RES = ANDROID_SRC / "res"
results = []
def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))
def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""
print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()
assert ANDROID.exists(), "android/ directory must exist"

Tower 21: Tower of Android


In [2]:
# === Cell 1: Capacitor project structure ===

# P01: android/ directory exists
record("T21-P01", "android/ directory exists",
       ANDROID.is_dir())

# P02: capacitor config exists at project root (.ts or .json)
cap_config_ts = BROWSER_ROOT / "capacitor.config.ts"
cap_config_json = BROWSER_ROOT / "capacitor.config.json"
if cap_config_ts.exists():
    cap_config_path = cap_config_ts
    cap_config = read_file(cap_config_ts)
    cap_format = "ts"
elif cap_config_json.exists():
    cap_config_path = cap_config_json
    cap_config = read_file(cap_config_json)
    cap_format = "json"
else:
    cap_config_path = cap_config_ts
    cap_config = ""
    cap_format = "missing"
record("T21-P02", "capacitor config exists at project root (.ts or .json)",
       len(cap_config) > 0,
       f"{cap_config_path.name} ({len(cap_config)} bytes, format={cap_format})")

# P03: appId is com.solaceagi.browser in capacitor config
record("T21-P03", "capacitor appId = com.solaceagi.browser",
       'com.solaceagi.browser' in cap_config,
       f"found in {cap_config_path.name}")

# P04: webDir points to 'web' in capacitor config
# JSON uses "webDir": "web", TS may use webDir: 'web'
record("T21-P04", "capacitor webDir = web",
       '"webDir"' in cap_config and '"web"' in cap_config or
       "'webDir'" in cap_config and "'web'" in cap_config or
       'webDir:' in cap_config and "'web'" in cap_config,
       f"webDir configured in {cap_config_path.name}")

# P05: @capacitor/android in package.json
pkg_json = read_file(BROWSER_ROOT / "package.json")
record("T21-P05", "@capacitor/android dependency in package.json",
       '@capacitor/android' in pkg_json)

# P06: @capacitor/core in package.json
record("T21-P06", "@capacitor/core dependency in package.json",
       '@capacitor/core' in pkg_json)

# P07: androidScheme is https in capacitor config (secure WebView)
record("T21-P07", "capacitor android scheme = https",
       'androidScheme' in cap_config and 'https' in cap_config,
       f"androidScheme configured in {cap_config_path.name}")

PASS: android/ directory exists
PASS: capacitor config exists at project root (.ts or .json) -- capacitor.config.json (580 bytes, format=json)
PASS: capacitor appId = com.solaceagi.browser -- found in capacitor.config.json
PASS: capacitor webDir = web -- webDir configured in capacitor.config.json
PASS: @capacitor/android dependency in package.json
PASS: @capacitor/core dependency in package.json
PASS: capacitor android scheme = https -- androidScheme configured in capacitor.config.json


In [3]:
# === Cell 2: Android Gradle and build config ===

# P08: app/build.gradle exists
build_gradle = read_file(ANDROID_APP / "build.gradle")
record("T21-P08", "app/build.gradle exists",
       len(build_gradle) > 0)

# P09: applicationId matches Capacitor appId
record("T21-P09", "applicationId = com.solaceagi.browser in build.gradle",
       'com.solaceagi.browser' in build_gradle)

# P10: variables.gradle defines SDK versions
variables = read_file(ANDROID / "variables.gradle")
has_min_sdk = 'minSdkVersion' in variables
has_compile_sdk = 'compileSdkVersion' in variables
has_target_sdk = 'targetSdkVersion' in variables
record("T21-P10", "variables.gradle defines minSdk/compileSdk/targetSdk",
       has_min_sdk and has_compile_sdk and has_target_sdk,
       f"min={has_min_sdk}, compile={has_compile_sdk}, target={has_target_sdk}")

# P11: minSdkVersion >= 24 (Android 7.0 — reasonable for Capacitor)
min_sdk_match = re.search(r'minSdkVersion\s*=\s*(\d+)', variables)
min_sdk = int(min_sdk_match.group(1)) if min_sdk_match else 0
record("T21-P11", "minSdkVersion >= 24 (Android 7.0)",
       min_sdk >= 24,
       f"minSdkVersion = {min_sdk}")

# P12: proguard-rules.pro exists
proguard = read_file(ANDROID_APP / "proguard-rules.pro")
record("T21-P12", "proguard-rules.pro exists",
       len(proguard) > 0)

PASS: app/build.gradle exists
PASS: applicationId = com.solaceagi.browser in build.gradle
PASS: variables.gradle defines minSdk/compileSdk/targetSdk -- min=True, compile=True, target=True
PASS: minSdkVersion >= 24 (Android 7.0) -- minSdkVersion = 24
PASS: proguard-rules.pro exists


In [4]:
# === Cell 3: AndroidManifest and Activity ===

# P13: AndroidManifest.xml exists
manifest = read_file(ANDROID_SRC / "AndroidManifest.xml")
record("T21-P13", "AndroidManifest.xml exists",
       len(manifest) > 0)

# P14: INTERNET permission declared
record("T21-P14", "INTERNET permission declared in manifest",
       'android.permission.INTERNET' in manifest)

# P15: MainActivity exists and extends BridgeActivity
main_activity = read_file(ANDROID_SRC / "java" / "com" / "solaceagi" / "browser" / "MainActivity.java")
record("T21-P15", "MainActivity.java exists and extends BridgeActivity",
       'BridgeActivity' in main_activity,
       "Capacitor BridgeActivity pattern")

# P16: Manifest has LAUNCHER intent filter
record("T21-P16", "LAUNCHER intent filter in manifest",
       'android.intent.category.LAUNCHER' in manifest)

# P17: supportsRtl is true in manifest
record("T21-P17", "supportsRtl=true in manifest",
       'android:supportsRtl="true"' in manifest)

PASS: AndroidManifest.xml exists
PASS: INTERNET permission declared in manifest
PASS: MainActivity.java exists and extends BridgeActivity -- Capacitor BridgeActivity pattern
PASS: LAUNCHER intent filter in manifest
PASS: supportsRtl=true in manifest


In [5]:
# === Cell 4: Icons, splash screens, and resources ===

# P18: Launcher icons exist in all density buckets
densities = ['mdpi', 'hdpi', 'xhdpi', 'xxhdpi', 'xxxhdpi']
all_icons = all(
    (ANDROID_RES / f"mipmap-{d}" / "ic_launcher.png").exists()
    for d in densities
)
record("T21-P18", "Launcher icons in all 5 density buckets",
       all_icons,
       f"checked: {densities}")

# P19: Adaptive icon XML exists (Android 8+)
adaptive_icon = ANDROID_RES / "mipmap-anydpi-v26" / "ic_launcher.xml"
record("T21-P19", "Adaptive icon XML exists (v26+)",
       adaptive_icon.exists())

# P20: Splash screens exist for portrait AND landscape
port_splashes = [d for d in ANDROID_RES.iterdir() if d.name.startswith('drawable-port-') and (d / 'splash.png').exists()]
land_splashes = [d for d in ANDROID_RES.iterdir() if d.name.startswith('drawable-land-') and (d / 'splash.png').exists()]
record("T21-P20", "Splash screens for portrait and landscape densities",
       len(port_splashes) >= 5 and len(land_splashes) >= 5,
       f"portrait={len(port_splashes)}, landscape={len(land_splashes)}")

# P21: strings.xml has app_name
strings_xml = read_file(ANDROID_RES / "values" / "strings.xml")
record("T21-P21", "strings.xml has app_name = Solace Browser",
       'Solace Browser' in strings_xml)

# P22: styles.xml has SplashScreen theme
styles_xml = read_file(ANDROID_RES / "values" / "styles.xml")
record("T21-P22", "styles.xml has SplashScreen theme",
       'SplashScreen' in styles_xml or 'Theme.SplashScreen' in styles_xml)

PASS: Launcher icons in all 5 density buckets -- checked: ['mdpi', 'hdpi', 'xhdpi', 'xxhdpi', 'xxxhdpi']
PASS: Adaptive icon XML exists (v26+)
PASS: Splash screens for portrait and landscape densities -- portrait=5, landscape=5
PASS: strings.xml has app_name = Solace Browser
PASS: styles.xml has SplashScreen theme


In [6]:
# === Cell 5: PWA manifest and service worker (Android web layer) ===

# P23: manifest.json exists with correct fields
manifest_json = read_file(WEB / "manifest.json")
has_manifest = len(manifest_json) > 0
if has_manifest:
    mdata = json.loads(manifest_json)
else:
    mdata = {}
record("T21-P23", "web/manifest.json exists and is valid JSON",
       has_manifest and bool(mdata))

# P24: manifest has display: standalone
record("T21-P24", "manifest display = standalone",
       mdata.get('display') == 'standalone')

# P25: manifest has Play Store related_applications
related_apps = mdata.get('related_applications', [])
play_app = any(a.get('platform') == 'play' for a in related_apps)
record("T21-P25", "manifest has Play Store related_applications",
       play_app)

# P26: manifest icons include 192 and 512 (PWA requirements)
icon_sizes = [i.get('sizes', '') for i in mdata.get('icons', [])]
has_192 = any('192' in s for s in icon_sizes)
has_512 = any('512' in s for s in icon_sizes)
record("T21-P26", "manifest has 192x192 and 512x512 icons",
       has_192 and has_512,
       f"192={has_192}, 512={has_512}")

# P27: service worker exists
sw_content = read_file(WEB / "sw.js")
record("T21-P27", "sw.js service worker exists",
       len(sw_content) > 50)

PASS: web/manifest.json exists and is valid JSON
PASS: manifest display = standalone
PASS: manifest has Play Store related_applications
PASS: manifest has 192x192 and 512x512 icons -- 192=True, 512=True
PASS: sw.js service worker exists


In [7]:
# === Cell 6: Responsive design and touch targets ===

# P28: site.css has mobile breakpoints
site_css = read_file(WEB / "css" / "site.css")
mobile_breakpoints = re.findall(r'@media\s*\(max-width:\s*(\d+)px\)', site_css)
mobile_bp_values = [int(v) for v in mobile_breakpoints]
# Android phones: typically < 768px, some < 480px
has_phone_bp = any(v <= 768 for v in mobile_bp_values)
record("T21-P28", "site.css has mobile breakpoints (<= 768px)",
       has_phone_bp,
       f"breakpoints found: {sorted(set(mobile_bp_values))}")

# P29: Touch targets >= 44px in CSS
touch_44 = re.findall(r'min-(?:height|width):\s*4[4-9]px', site_css)
record("T21-P29", "Touch targets >= 44px defined in site.css",
       len(touch_44) >= 2,
       f"{len(touch_44)} occurrences of min-height/width >= 44px")

# P30: viewport meta tag in main HTML pages
key_pages = ['home.html', 'app-store.html', 'settings.html', 'schedule.html', 'start.html']
pages_with_viewport = 0
for page in key_pages:
    content = read_file(WEB / page)
    if 'name="viewport"' in content:
        pages_with_viewport += 1
record("T21-P30", "viewport meta tag in key HTML pages",
       pages_with_viewport == len(key_pages),
       f"{pages_with_viewport}/{len(key_pages)} pages have viewport meta")

PASS: site.css has mobile breakpoints (<= 768px) -- breakpoints found: [500, 540, 600, 680, 720, 767, 768, 900, 980]
PASS: Touch targets >= 44px defined in site.css -- 16 occurrences of min-height/width >= 44px
PASS: viewport meta tag in key HTML pages -- 5/5 pages have viewport meta


In [8]:
# === Evidence Summary ===
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 21: Tower of Android
Probes: 30 | Passed: 30 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 5b0b441d80cb3241

{
  "surface": "android-t21-qa",
  "tower": 21,
  "tower_name": "Tower of Android",
  "timestamp": "2026-03-06T11:34:00.532601",
  "total_probes": 30,
  "passed": 30,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "5b0b441d80cb3241"
}
